# Step 7 — Retrieval tools (the "read" path)

The ingestion pipeline **writes** into Neo4j. This step builds the tools that **read** from it — the same tools the agentic RAG chatbot (Step 8) will call.

| Tool | What it answers | Mechanism |
|---|---|---|
| `vector_search` | "What passages are *about* this idea?" | Cosine similarity on chunk embeddings |
| `fulltext_search` | "Where is this exact keyword / CVE / docket?" | Lucene full-text index (+ document id/title fallback) |
| `graph_search` | "What do we know about this entity, and how is it linked?" | Entity match → `RELATED_TO` neighbors → `MENTIONS` provenance chunks |

Every hit returns **evidence** (passage text) and **source metadata** (title, agency, domain, URL) so answers can be cited.

> Code lives in `signalpulse/retrieval.py`.

## 1. Concepts

**Why three tools?** No single search mode wins every question:
- Semantic ("cybersecurity risk management") → vector
- Exact id (`CVE-2026-56291`) → fulltext / metadata
- Relational ("what is CISA connected to?") → graph

**Citations.** Each `Evidence` object carries `title`, `agency`, `url`, and `text`. The agent must ground claims in these passages and link the URL — that is the anti-hallucination contract.

**Similarity threshold.** Vector search drops hits below `SIMILARITY_THRESHOLD` (default 0.60). If nothing clears the bar, the agent should say the sources don't cover the question instead of guessing.

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from signalpulse import graph
from signalpulse import retrieval as R
from signalpulse.config import settings

graph.verify_connectivity()
print("Neo4j ready:", graph.node_counts())
print(f"VECTOR_TOP_K={settings.VECTOR_TOP_K}, SIMILARITY_THRESHOLD={settings.SIMILARITY_THRESHOLD}")

Neo4j ready: {'Document': 20, 'Chunk': 29, 'Entity': 122}
VECTOR_TOP_K=6, SIMILARITY_THRESHOLD=0.6


## 2. `vector_search` — semantic retrieval

1. Embed the question with the same local model (`bge-small-en-v1.5`), using the query instruction prefix.
2. Ask Neo4j's `chunk_embedding` vector index for the nearest neighbors (cosine).
3. Join each chunk back to its `Document` for title / agency / URL.
4. Drop scores below the threshold.

Optional `domain=` filter restricts results to one outline domain.

In [2]:
hits = R.vector_search(
    "How should organizations manage cybersecurity risk?",
    top_k=3,
    threshold=0.50,  # slightly looser for a small demo corpus
)
print(R.format_evidence(hits))

C:\Users\saihaj\Documents\22nd Project\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


[vector_search] score=0.862 | NIST | NIST.CSWP.29.pdf
  cite: https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf
  text: T he NIST Cybersecurity Framework (CSF) 2.0 i NIST C SWP 29 February 26, 2024 Abstract T he NIST Cybersecurity Framework (CSF) 2.0 provides guidance to industry, government agencies, and other organiz...

[vector_search] score=0.841 | NIST | NIST Risk Management Framework | CSRC
  cite: https://csrc.nist.gov/projects/risk-management
  text: June 30, 2026: NIST SP 800-18r2 (Revision 2), Developing Security, Privacy, and Cybersecurity Supply Chain Risk Management Plans for Systems, has been finalized and is now available. This revision bro...

[vector_search] score=0.812 | NIST | NIST SP 800-53 ac-1: Policy and Procedures
  cite: https://csrc.nist.gov/publications/detail/sp/800-53/rev-5/final
  text: NIST SP 800-53 control ac-1: Policy and Procedures. Family: Access Control. Access control policy and procedures address the controls in the AC family that are implem

## 3. `fulltext_search` — keyword / identifier retrieval

Uses Neo4j's Lucene `chunk_fulltext` index. Identifier-like queries (`CVE-2026-56291`) are rewritten to a stricter Lucene form, and we also match `Document.id` / `Document.title` because some sources put the id in metadata more reliably than in the passage body.

In [3]:
hits = R.fulltext_search("CVE-2026-56291", top_k=3)
print(R.format_evidence(hits))

print("\n--- another keyword: Account Management (NIST control theme) ---\n")
hits2 = R.fulltext_search("Account Management", top_k=2)
print(R.format_evidence(hits2))

[fulltext_search] score=1.500 | CISA | CVE-2026-56291: Balbooa Forms Unrestricted Upload 
  cite: https://nvd.nist.gov/vuln/detail/CVE-2026-56291
  text: . Stakeholders are responsible for evaluating each asset's internet exposure and ensuring adherence to BOD 26-04 patching guidelines. Due date: 2026-07-13.

[fulltext_search] score=1.500 | CISA | CVE-2026-56291: Balbooa Forms Unrestricted Upload 
  cite: https://nvd.nist.gov/vuln/detail/CVE-2026-56291
  text: Balbooa Forms Unrestricted Upload of File with Dangerous Type Vulnerability. Vendor: Balbooa. Product: Forms. Balbooa Forms contains an unrestricted upload of file with dangerous type vulnerability th...

--- another keyword: Account Management (NIST control theme) ---

[fulltext_search] score=2.352 | NIST | NIST SP 800-53 ac-2: Account Management
  cite: https://csrc.nist.gov/publications/detail/sp/800-53/rev-5/final
  text: NIST SP 800-53 control ac-2: Account Management. Family: Access Control. Examples of system account types

## 4. `graph_search` — entity & relationship lookup

1. Find `Entity` nodes whose name contains the query (case-insensitive).
2. Collect directed `RELATED_TO` triples involving that entity.
3. Follow `(:Chunk)-[:MENTIONS]->(:Entity)` and join to `Document` for **provenance / citations**.

This is what powers relational questions ("What does CISA connect to in our graph?").

In [4]:
for hit in R.graph_search("CISA", top_k=3):
    print(hit.preview())
    print()

print("--- NIST entity neighborhood ---\n")
for hit in R.graph_search("NIST", top_k=2):
    print(hit.preview())
    print()

Entity: CISA [Organization]
  (CISA) -ISSUED_BY-> (BOD 26-04)
  provenance chunks: 2
  [graph_search] score=1.000 | CISA | CVE-2026-48939: iCagenda Unrestricted Upload of Fi
    cite: https://nvd.nist.gov/vuln/detail/CVE-2026-48939
    text: iCagenda Unrestricted Upload of File with Dangerous Type Vulnerability. Vendor: iCagenda. Product: iCagenda. iCagenda co...
  [graph_search] score=1.000 | CISA | CVE-2026-56291: Balbooa Forms Unrestricted Upload 
    cite: https://nvd.nist.gov/vuln/detail/CVE-2026-56291
    text: Balbooa Forms Unrestricted Upload of File with Dangerous Type Vulnerability. Vendor: Balbooa. Product: Forms. Balbooa Fo...

--- NIST entity neighborhood ---

Entity: NIST [Organization]
  (NIST) -ISSUED_BY-> (NIST SP 800-18r2)
  (NIST) -ISSUED_BY-> (NIST SP 800-53)
  (NIST) -ISSUED_BY-> (NIST SP 800-53B)
  (NIST) -ISSUED_BY-> (NIST SP 800-53A)
  (NIST) -ISSUED_BY-> (NIST Cybersecurity Framework (CSF) 2.0)
  (Raman) -CONFIRMED_AT-> (NIST)
  provenance chunks: 2
  [graph_se

## 5. Optional domain filter

Both vector and fulltext accept `domain=` to stay inside one outline domain (still the same shared graph).

In [5]:
hits = R.vector_search(
    "artificial intelligence priorities for state government",
    top_k=3,
    threshold=0.40,
    domain="State & Local Gov",
)
print(R.format_evidence(hits))

[vector_search] score=0.866 | NASCIO | NASCIO 2026 State CIO Top Ten Priorities
  cite: https://www.nascio.org/resource/state-cio-top-ten-policy-and-technology-priorities-for-2026/
  text: Headline: Artificial intelligence takes the number one spot for the first time, overtaking cybersecurity, which held the top position for 12 straight years. Other shifts include upward movement for ac...

[vector_search] score=0.848 | NASCIO | NASCIO 2026 State CIO Top Ten Priorities
  cite: https://www.nascio.org/resource/state-cio-top-ten-policy-and-technology-priorities-for-2026/
  text: NASCIO State CIO Top Ten Policy and Technology Priorities for 2026 Source: National Association of State Chief Information Officers (NASCIO). Public summary of the annual State CIO Top 10 Priorities s...

## Recap & what's next

We now have the three retrieval tools the agent needs:

- **`vector_search`** — meaning-based chunk retrieval with citation metadata  
- **`fulltext_search`** — keyword / id retrieval (Lucene + document metadata)  
- **`graph_search`** — entity neighborhood + provenance chunks  

**Next — Step 8:** build the **agentic RAG** loop (LangGraph): the LLM chooses which tool(s) to call, reads the evidence, and answers with mandatory citations — or refuses if nothing clears the similarity threshold.

> Prompt to continue:
> *"Step 8: Create notebook `08_agent.ipynb`. Build the agentic RAG chatbot with LangGraph that uses vector_search, fulltext_search, and graph_search as tools. Enforce citations and a no-evidence guardrail. Demo a few questions end-to-end."*